In [1]:
import asyncio
import itertools
from typing import List
from benchmarks import benchmark_orchestrator
from benchmarks.answer_generators import (
    AdkAnswerGenerator,
    GeminiAnswerGenerator,
    GroundTruthAnswerGenerator,
    TrivialAnswerGenerator,
)
from benchmarks.data_models import BenchmarkRunResult
import pandas as pd

# Set pandas display options
pd.set_option('display.max_colwidth', None)

# ANSI escape codes for colors
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

def permute(cls, **kwargs):
    """Helper to generate permutations of class instances."""
    keys = kwargs.keys()
    values = kwargs.values()
    for instance_values in itertools.product(*values):
        yield cls(**dict(zip(keys, instance_values)))

# Read context from llms.txt
try:
    with open("llms.txt", "r", encoding="utf-8") as f:
        llms_context = f.read()
except FileNotFoundError:
    print(f"{bcolors.WARNING}Warning: llms.txt not found. Proceeding without context.{bcolors.ENDC}")
    llms_context = ""

async def run_comparison() -> List[BenchmarkRunResult]:
    """Sets up and runs the benchmark comparison."""
    print("Configuring benchmark run...")
    
    benchmark_suites = [
        "benchmarks/benchmark_definitions/api_understanding/benchmark.yaml",
        "benchmarks/benchmark_definitions/fix_errors/benchmark.yaml",
        "benchmarks/benchmark_definitions/multiple_choice/benchmark.yaml",
        "benchmarks/benchmark_definitions/advanced_adk_usage/benchmark.yaml",
        "benchmarks/benchmark_definitions/skill_based/benchmark.yaml",
    ]
    
    answer_generators = [
        GroundTruthAnswerGenerator(),
        TrivialAnswerGenerator(),
        *permute(
            GeminiAnswerGenerator,
            model_name=["gemini-2.5-flash", "gemini-2.5-pro"],
            context=[None, llms_context]
        ),
        # AdkAnswerGenerator(),
    ]
    
    print("Executing benchmarks...")
    results = await benchmark_orchestrator.run_benchmarks(
        benchmark_suites=benchmark_suites, answer_generators=answer_generators
    )
    
    return results

def analyze_logs(
    results_df: pd.DataFrame, generator_name: str, result_type: str = 'fail'
) -> None:
    """Filters and displays benchmark results for a specific generator and result type."""
    
    result_value = 1 if result_type.lower() == 'pass' else 0
    
    print(f"{bcolors.HEADER}--- Analyzing {result_type.upper()}S for {generator_name} ---{bcolors.ENDC}")
    
    filtered_df = results_df[
        (results_df['answer_generator'] == generator_name) & 
        (results_df['result'] == result_value)
    ]
    
    if filtered_df.empty:
        print(f"{bcolors.OKGREEN}No {result_type}s found for {generator_name}.{bcolors.ENDC}")
        return
    
    for _, row in filtered_df.iterrows():
        print(f"{bcolors.WARNING}Suite: {row['suite']}{bcolors.ENDC}")
        print(f"{bcolors.WARNING}Benchmark: {row['benchmark_name']}{bcolors.ENDC}")
        print(f"{bcolors.OKCYAN}  Answer:{bcolors.ENDC}\n    {row['answer']}")
        if result_type.lower() == 'fail':
            print(f"{bcolors.FAIL}  Validation Error:{bcolors.ENDC}\n    {row['validation_error']}")
            if "temp_test_file" in row and pd.notna(row["temp_test_file"]):
                print(f"{bcolors.OKBLUE}  Temp File:{bcolors.ENDC} {row['temp_test_file']}")
        print("-" * 40)


In [2]:
# Execute the benchmarks
results = await run_comparison()
raw_results_df = pd.DataFrame([r.model_dump() for r in results])

# Calculate summary from raw results
summary_df = (
  raw_results_df.groupby("answer_generator")
  .agg(
      passed=("result", "sum"),
      total=("result", "count"),
      mean_latency=("latency", "mean"),
      p50_latency=("latency", lambda x: x.quantile(0.5)),
      p90_latency=("latency", lambda x: x.quantile(0.9)),
      p99_latency=("latency", lambda x: x.quantile(0.99)),
  )
)
summary_df["pass_rate"] = summary_df["passed"] / summary_df["total"]

print(f"{bcolors.HEADER}--- Benchmark Summary ---{bcolors.ENDC}")
# For displaying in a script, you might want to use print(summary_df) instead of print(summary_df)
print(summary_df)

# --- Analysis Configuration ---
generator_to_analyze = 'GeminiAnswerGenerator(gemini-2.5-pro-with-context)' 
result_type_to_see = 'fail' 

analyze_logs(
    results_df=raw_results_df,
    generator_name=generator_to_analyze,
    result_type=result_type_to_see
)


Configuring benchmark run...
Executing benchmarks...
--- Loading benchmark suite: benchmarks/benchmark_definitions/api_understanding/benchmark.yaml ---
  - Queuing tests for answer generator: GroundTruthAnswerGenerator
  - Queuing tests for answer generator: TrivialAnswerGenerator
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash-with-context)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-pro)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-pro-with-context)
--- Loading benchmark suite: benchmarks/benchmark_definitions/fix_errors/benchmark.yaml ---
  - Queuing tests for answer generator: GroundTruthAnswerGenerator
  - Queuing tests for answer generator: TrivialAnswerGenerator
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-f

100%|█████████████████████████████████████████████████████████████████| 738/738 [01:37<00:00,  7.57it/s]

--- Benchmark Summary ---
                                                      passed  total  \
answer_generator                                                      
GeminiAnswerGenerator(gemini-2.5-flash)                   65    123   
GeminiAnswerGenerator(gemini-2.5-flash-with-context)      71    123   
GeminiAnswerGenerator(gemini-2.5-pro)                     63    123   
GeminiAnswerGenerator(gemini-2.5-pro-with-context)        64    123   
GroundTruthAnswerGenerator                               120    123   
TrivialAnswerGenerator                                    15    123   

                                                      mean_latency  \
answer_generator                                                     
GeminiAnswerGenerator(gemini-2.5-flash)                   9.559865   
GeminiAnswerGenerator(gemini-2.5-flash-with-context)     11.409500   
GeminiAnswerGenerator(gemini-2.5-pro)                    16.208952   
GeminiAnswerGenerator(gemini-2.5-pro-with-context)     